In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
# Load market data
market = pd.read_csv("../../data/processed/market_all_features.csv", parse_dates=["date"])
market["ticker"] = market["ticker"].astype(str)

print("Market shape:", market.shape)
display(market.head())


In [ ]:
# Load sentiment (Parquet is preferred)
sent = pd.read_parquet("../../data/processed/sentiment_daily.parquet")
sent["date"] = pd.to_datetime(sent["date"])
sent["ticker"] = sent["ticker"].astype(str)

print("Sentiment shape:", sent.shape)
display(sent.head())


In [ ]:
sent["sent_cat_lag1"] = (
    sent.groupby("ticker")["sent_daily_cat"].shift(1)
)

sent = sent[
    [
        "ticker",
        "date",
        "sent_mean",
        "sent_std",
        "sent_count",
        "sent_pos",
        "sent_neg",
        "sent_neu",
        "sent_daily_cat",
        "sent_lag1",
        "sent_cat_lag1",
        "sent_roll3"
    ]
]
display(sent.head())

In [ ]:
sent.to_parquet("../../data/processed/sentiment_daily_category_shifted.parquet")
sent.to_csv("../../data/processed/sentiment_daily_category_shifted.csv", index=False)

In [ ]:
# Merge
features = market.merge(
    sent,
    on=["ticker", "date"],
    how="left",    # keep ALL market rows (important!)
    validate="many_to_one"  # each (ticker, date) in sentiment appears once
)

In [ ]:
print("Merged shape:", features.shape)
features.head()

In [ ]:
# 1. Sort by ticker/date (always good for time series)
features = features.sort_values(["ticker", "date"]).reset_index(drop=True)

# 2. Drop rows where return is NaN (first day per ticker)
before = features.shape[0]
features = features[~features["return"].isna()].copy()
after = features.shape[0]
print(f"Dropped {before - after} rows with NaN return.")

# 3. Create stable row_id
features["row_id"] = features["ticker"] + "|" + features["date"].astype(str)

# 4. Fill sentiment NaNs for no-news days
sent_cols_num = [
    "sent_mean", "sent_std", "sent_count",
    "sent_pos", "sent_neg", "sent_neu",
    "sent_lag1", "sent_roll3",
]

for col in sent_cols_num:
    if col in features.columns:
        features[col] = features[col].fillna(0)

if "sent_daily_cat" in features.columns:
    features["sent_daily_cat"] = features["sent_daily_cat"].fillna("neutral")

features.head()


In [ ]:
import pandas as pd
import numpy as np

# 1) Make sure date is datetime and data is sorted
features["date"] = pd.to_datetime(features["date"])
features = features.sort_values(["date", "ticker"]).reset_index(drop=True)

# 2) Choose a clear calendar cutoff for train/test
# 👉 Adjust this if you want a different boundary
split_date = pd.Timestamp("2023-01-01")

# 3) Build masks
train_mask = features["date"] < split_date
test_mask  = features["date"] >= split_date

# 4) Add split columns (these will be reused for ALL models)
features["is_train"] = train_mask
features["set"] = np.where(features["is_train"], "train", "test")

# 5) Quick sanity checks
print(features["set"].value_counts())
print(features.groupby("set")["date"].agg(["min", "max"]))


In [ ]:
features.head()

In [ ]:
features.to_parquet("../../data/processed/market_all_features_with_sentiment.parquet")
features.to_csv("../../data/processed/market_all_features_with_sentiment.csv", index=False)